# healthcare_US — HCDE 530, Week 5

**CDC NHSN Hospital Respiratory Data (HRD)** — preliminary metrics by jurisdiction and week — loaded from the CSV in this folder.

The file has many columns (beds, occupancy, COVID / flu / RSV hospitalizations and admissions, reporting coverage, per-capita rates). Rows combine a **week ending date** with a **geographic aggregation** code (e.g. `USA` for national totals).

## Setup

In [25]:
from pathlib import Path

import pandas as pd

DATA_FILE = "Weekly_Hospital_Respiratory_Data.csv"
assert Path(DATA_FILE).exists(), f"Missing file: {DATA_FILE}"

## Load the CSV

Read **`Weekly_Hospital_Respiratory_Data.csv`** into `hrd`, then inspect with **`head()`** (first rows) and **`info()`** (column names, dtypes, non-null counts).

Numeric fields often use comma thousands separators (e.g. `716,330.11`). Use `thousands=","` when reading. Percent columns stay as strings until you strip `%` if you need floats.

In [26]:
hrd = pd.read_csv(DATA_FILE, thousands=",")
# Question: how many rows and columns are there, and what do a few example rows look like?
# Answer: shape is the table size; head() previews real values so we see column names, types, and lots of NaNs for non-US rows early on.
print(hrd.shape)
hrd.head()

(17956, 190)


,Week Ending Date,Geographic aggregation,Number of Inpatient Beds,Number of Adult Inpatient Beds,Number of Pediatric Inpatient beds,Number of Inpatient Beds Occupied,Number of Adult Inpatient Beds Occupied,Number of Pediatric Inpatient Beds Occupied,Number of ICU Beds,Number of Adult ICU Beds,...,"Number of Adult RSV Admissions, 65-74 years, per 100,000 population","Number of Adult RSV Admissions, 75+ years, per 100,000 population","Total Number of Adult RSV Admissions per 100,000 population","Total number of RSV Admissions per 100,000 population",Percent Hospitals Reporting Total New COVID-19 Admissions Above 80%,Percent Hospitals Reporting Total New COVID-19 Admissions Above 90%,Percent Hospitals Reporting Total New Influenza Admissions Above 80%,Percent Hospitals Reporting Total New Influenza Admissions Above 90%,Percent Hospitals Reporting Total New RSV Admissions Above 80%,Percent Hospitals Reporting Total New RSV Admissions Above 90%
0,2023-12-30,MP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0
1,2021-03-20,AS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0
2,2024-07-27,MA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0
3,2020-08-22,AS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0
4,2023-03-11,MP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0


In [27]:
# Question: for each column, what dtype is stored and how many non-null entries exist?
# Answer: info() tells us which fields are numeric vs object (e.g. percent strings) and how sparse wide HRD columns are before analysis.
hrd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17956 entries, 0 to 17955
Columns: 190 entries, Week Ending Date to Percent Hospitals Reporting Total New RSV Admissions Above 90%
dtypes: float64(87), int64(33), object(70)
memory usage: 26.0+ MB


## Jurisdictions and national time series

Filter `Geographic aggregation == "USA"` for national weekly rows, then sort by `Week Ending Date`.

In [28]:
# Question: which geographic codes appear most often in this file?
# Answer: value_counts() ranks how many rows each code has—here many jurisdictions tie because the file is weekly with one row per area per week.
hrd["Geographic aggregation"].value_counts().head(15)

Geographic aggregation
MP    268
AL    268
MD    268
ME    268
MO    268
MT    268
NC    268
VT    268
DE    268
IA    268
IN    268
ND    268
NM    268
TN    268
WA    268
Name: count, dtype: int64

In [29]:
# Question: what if we only keep national (USA) weekly rows instead of every state/territory?
# Answer: equality on Geographic aggregation gives one US-wide row per week—our main “meaningful subset” for trends.
usa = (
    hrd.loc[hrd["Geographic aggregation"] == "USA"]
    .assign(**{"Week Ending Date": lambda d: pd.to_datetime(d["Week Ending Date"])})
    .sort_values("Week Ending Date")
    .reset_index(drop=True)
)
print(usa.shape)
usa[["Week Ending Date", "Number of Inpatient Beds", "Percent Inpatient Beds Occupied",
     "Total COVID-19 Admissions", "Total Influenza Admissions", "Total RSV Admissions"]].tail(8)

# Question: on how many national weeks were reported inpatient beds above 700,000?
# Answer: a strict df[df["column"] > value] filter counts high-reported-capacity weeks (numeric threshold subset from class).
usa_high_beds = usa[usa["Number of Inpatient Beds"] > 700_000]
print("Weeks with inpatient beds > 700,000:", len(usa_high_beds))

(268, 190)
Weeks with inpatient beds > 700,000: 141


### Q1 — What's the distribution of your most important column?

For hospital strain, **percent inpatient beds occupied** is a central summary. Below uses national (`USA`) weekly rows and parses the `%` strings into numbers, then **summary stats** and a **binned frequency table** (same idea as a histogram, no extra libraries).

Run **Setup** (imports `pandas` as `pd`), then the cells that load `hrd` and build `usa`. Q1 uses only pandas — no `matplotlib` install required. If you later want charts, install with: `python -m pip install matplotlib` (in the same environment as your Jupyter kernel).

In [30]:
# Question: how spread out is national inpatient occupancy across weeks (our main strain indicator)?
# Answer: describe() gives center/spread/min/max; binned counts show whether weeks cluster in a narrow band or span wide ranges.
col = "Percent Inpatient Beds Occupied"
pct_inpatient_occ = pd.to_numeric(
    usa[col].astype(str).str.replace("%", "", regex=False).str.strip(),
    errors="coerce",
)
print(f"Column: {col}")
print(pct_inpatient_occ.describe())

# Same distribution as a table: each row is how many weeks fell in that % range (no chart library needed).
bins = pd.cut(pct_inpatient_occ.dropna(), bins=20, precision=2)
bin_counts = bins.value_counts().sort_index()
print("\nWeeks per bin (histogram as a table):")
display(bin_counts.to_frame("n_weeks"))

Column: Percent Inpatient Beds Occupied
count    268.000000
mean      74.245261
std        2.522152
min       64.960000
25%       72.997500
50%       74.505000
75%       76.042500
max       79.260000
Name: Percent Inpatient Beds Occupied, dtype: float64

Weeks per bin (histogram as a table):


,n_weeks
Percent Inpatient Beds Occupied,
"(64.95, 65.68]",1
"(65.68, 66.39]",0
"(66.39, 67.1]",2
"(67.1, 67.82]",0
"(67.82, 68.54]",8
"(68.54, 69.25]",1
"(69.25, 69.96]",7
"(69.96, 70.68]",8
"(70.68, 71.4]",7


### Q2 — Filter to a meaningful subset. What's in it?

**Subset:** only **national** rows (`Geographic aggregation == "USA"`), sorted by week — the `usa` table from above. Each row is one **week ending date** with US-wide bed, occupancy, and pathogen admission metrics (not individual hospitals).

In [31]:
# Question: inside the USA-only subset, what date span and summary stats do key numeric columns cover?
# Answer: shape/date range say how much history we have; describe() summarizes typical bed counts and admissions in that national slice.
subset = usa
print("Filter: Geographic aggregation == 'USA'")
print("Shape (rows, columns):", subset.shape)
print(
    "Week range:",
    subset["Week Ending Date"].min().date(),
    "→",
    subset["Week Ending Date"].max().date(),
)
print("\nFirst / last 3 rows (selected columns):")
display(
    subset[
        [
            "Week Ending Date",
            "Number of Inpatient Beds",
            "Percent Inpatient Beds Occupied",
            "Total COVID-19 Admissions",
            "Total Influenza Admissions",
            "Total RSV Admissions",
        ]
    ].iloc[list(range(3)) + list(range(-3, 0))],
)
subset[
    [
        "Number of Inpatient Beds",
        "Total COVID-19 Admissions",
        "Total Influenza Admissions",
        "Total RSV Admissions",
    ]
].describe()

Filter: Geographic aggregation == 'USA'
Shape (rows, columns): (268, 190)
Week range: 2020-08-08 → 2025-09-20

First / last 3 rows (selected columns):


,Week Ending Date,Number of Inpatient Beds,Percent Inpatient Beds Occupied,Total COVID-19 Admissions,Total Influenza Admissions,Total RSV Admissions
0,2020-08-08,773583.54,68.39%,38967.0,0.0,NaN
1,2020-08-15,779886.27,68.36%,36585.0,0.0,NaN
2,2020-08-22,794276.15,67.87%,36824.0,0.0,NaN
265,2025-09-06,697271.00,72.64%,10663.0,921.0,271.0
266,2025-09-13,686855.00,73.52%,9990.0,958.0,341.0
267,2025-09-20,652410.00,73.81%,8844.0,816.0,333.0


,Number of Inpatient Beds,Total COVID-19 Admissions,Total Influenza Admissions,Total RSV Admissions
count,268.000000,268.000000,268.000000,103.000000
mean,678352.752388,29820.615672,4204.958955,1757.514563
std,137439.612299,29381.127651,8550.369363,3296.903080
min,106716.000000,1978.000000,0.000000,0.000000
25%,686113.525000,8838.000000,460.750000,11.000000
50%,702698.005000,21167.500000,945.500000,233.000000
75%,736438.250000,38384.000000,2988.000000,1051.000000
max,832163.810000,156121.000000,54275.000000,15244.000000


### Q3 — Group by a category and find the average of a numeric column

**Category:** `Geographic aggregation` (jurisdiction code: each US state/territory plus `USA`).  
**Numeric column:** `Number of Inpatient Beds` — average reported beds per jurisdiction across all weeks in the file (states with sparse reporting will have more missing weeks reflected in the mean).

In [32]:
# Question: averaged across all weeks, which jurisdictions report the largest mean inpatient bed counts?
# Answer: groupby(...).mean() compares typical reported scale by area (USA totals top; sparse areas may drop out after dropna).
avg_beds_by_area = (
    hrd.groupby("Geographic aggregation", as_index=False)["Number of Inpatient Beds"]
    .mean()
    .dropna(subset=["Number of Inpatient Beds"])
    .sort_values("Number of Inpatient Beds", ascending=False)
)
print("Average Number of Inpatient Beds by jurisdiction (top 15):")
display(avg_beds_by_area.head(15))

Average Number of Inpatient Beds by jurisdiction (top 15):


,Geographic aggregation,Number of Inpatient Beds
58,USA,678352.752388
48,Region 4,157271.140299
49,Region 5,104892.469515
50,Region 6,90179.838209
53,Region 9,83159.149963
46,Region 2,70056.536642
47,Region 3,63335.891791
5,CA,59535.144328
57,TX,56635.117164
10,FL,51795.617873


### Q4 — Where are the missing values? Are any columns incomplete?

Summarize **missing count** and **% missing** per column. Many HRD fields are only filled for certain jurisdictions or reporting periods, so a large share of columns can be partly empty.

In [33]:
# Question: which columns are missing the most cells, and how incomplete is the sheet overall?
# Answer: isnull().sum() (same idea as isna) counts gaps per column—high counts mean that metric is often not reported for many rows/jurisdictions.
na_count = hrd.isnull().sum()
na_pct = 100 * hrd.isnull().mean()
n_cols = len(hrd.columns)
n_complete = int((na_count == 0).sum())
n_any_missing = int((na_count > 0).sum())
print(f"Columns with no missing values: {n_complete} / {n_cols}")
print(f"Columns with at least one missing: {n_any_missing} / {n_cols}")
print("\nTop 15 columns by % missing:")
display(na_pct.sort_values(ascending=False).head(15).round(2).to_frame("pct_missing"))
print("Top 15 columns by count of missing cells:")
display(na_count.sort_values(ascending=False).head(15).to_frame("n_missing"))

Columns with no missing values: 89 / 190
Columns with at least one missing: 101 / 190

Top 15 columns by % missing:


,pct_missing
"Number of Adult RSV Admissions, 75+ years",81.59
"Number of Adult Influenza Admissions, 75+ years",81.59
"Number of Pediatric RSV Admissions, 0-4 years, per 100,000 population",81.59
"Number of Adult RSV Admissions, 65-74 years",81.59
"Number of Adult RSV Admissions, 50-64 years",81.59
"Number of Adult RSV Admissions, 18-49 years",81.59
"Number of Pediatric RSV Admissions, 5-17 years",81.59
"Number of Pediatric RSV Admissions, 0-4 years",81.59
"Number of Influenza Admissions, unknown age",81.59
"Number of Adult Influenza Admissions, 65-74 years",81.59


Top 15 columns by count of missing cells:


,n_missing
"Number of Adult RSV Admissions, 75+ years",14650
"Number of Adult Influenza Admissions, 75+ years",14650
"Number of Pediatric RSV Admissions, 0-4 years, per 100,000 population",14650
"Number of Adult RSV Admissions, 65-74 years",14650
"Number of Adult RSV Admissions, 50-64 years",14650
"Number of Adult RSV Admissions, 18-49 years",14650
"Number of Pediatric RSV Admissions, 5-17 years",14650
"Number of Pediatric RSV Admissions, 0-4 years",14650
"Number of Influenza Admissions, unknown age",14650
"Number of Adult Influenza Admissions, 65-74 years",14650


## Next steps (optional)

- Parse percent columns: e.g. `usa["Percent Inpatient Beds Occupied"].str.rstrip("%").astype(float)`
- Plot a column over `Week Ending Date` with `matplotlib` or pandas `.plot()`
- Compare states by filtering other `Geographic aggregation` codes once you know the codebook